# Train the semantic whole-latent scheduler

**Experiment boundary:** 60 distinct LADI v2 images, 36/12/12 group-safe
split, five simulated UEs, whole SwinJSCC latent jobs. P2 and P3 are
image-derived; P5/P6 use the fixed task rubric and P7 is a prototype
default. These annotations are explicitly not human reviewed.
The DQN learns a *next-job decision*, not a one-shot permutation.

This notebook starts with a disposable clean clone of the latest remote
`semantic-aware-ai-scheduler` commit. Results go to a separate persistent
run directory. It requires a complete experiment-ready manifest and
measured payload when measured mode is selected.

## 1. Clean reset and exact source revision

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/saltypal/SemanticSchedulerEnd2End.git"
BRANCH = "semantic-aware-ai-scheduler"
IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ
WORKSPACE = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("SCHEDULER_WORKSPACE", Path.cwd())).resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = WORKSPACE / "project-src"
if PROJECT_ROOT.exists():
    if PROJECT_ROOT.resolve().parent != WORKSPACE:
        raise RuntimeError("Refusing to remove a checkout outside the disposable workspace")
    shutil.rmtree(PROJECT_ROOT)
subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_ROOT, check=True)
subprocess.run(["git", "checkout", "--detach", f"origin/{BRANCH}"], cwd=PROJECT_ROOT, check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
sys.path.insert(0, str(PROJECT_ROOT))
print({"environment": "kaggle" if IS_KAGGLE else "colab" if IS_COLAB else "local",
       "branch": BRANCH, "exact_commit": SOURCE_COMMIT, "checkout": str(PROJECT_ROOT)})

## 2. Dependencies, hardware, and persistent output

In [ ]:
import importlib.metadata as package_metadata
import json, platform
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "environment/requirements-semantic-scheduler-kaggle.txt")], check=True)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
if os.environ.get("TPU_NAME") or os.environ.get("XRT_TPU_CONFIG"):
    raise RuntimeError("This PyTorch DQN notebook has no TPU/XLA execution path; select CPU or CUDA")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENVIRONMENT = {
    "python": platform.python_version(), "platform": platform.platform(),
    "device": DEVICE, "gpu_count": torch.cuda.device_count(),
    "gpu_name": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "packages": {name: package_metadata.version(name) for name in
                 ("torch", "numpy", "pandas", "scikit-learn", "transformers", "datasets")},
}
print(ENVIRONMENT)

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from dataclasses import asdict
import yaml
from INFRA.ladi_manifest import validate_ladi60_manifest, render_contact_sheet
from INFRA.kaggle_dataset_bundle import resolve_ladi_manifest
from INFRA.semantic_config import load_scheduler_config, load_ue_profiles
from INFRA.experiments.semantic_scheduler_experiment import (
    TASK_ID, prepare_feature_cache, load_catalog, build_suites_from_manifest,
    run_provenance, simulated_fixed_payload_costs,
)
from INFRA.semantic_contracts import ResourceBudgetSemantic
from INFRA.learning.supervised_training import fit_supervised_models
from INFRA.learning.dqn_training import DQNConfig, train_dqn_seeds
from EncDec.integration.scheduler_replay import cache_swinjscc_payloads

RESULTS_ROOT = Path(os.environ.get("SCHEDULER_RESULTS_DIR",
    "/kaggle/working/outputs" if IS_KAGGLE else WORKSPACE / "semantic-scheduler-results"))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + SOURCE_COMMIT[:8]
RUN_ROOT = RESULTS_ROOT / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
MANIFEST_PATH = Path(os.environ.get("SCHEDULER_MANIFEST",
    "/kaggle/input/semantic-scheduler-ladi60-prototype/ladi60_semantic_scheduler_v1.csv"))
PAYLOAD_MODE = os.environ.get("SCHEDULER_PAYLOAD_MODE", "measured")
if PAYLOAD_MODE not in ("measured", "simulated_fixed"):
    raise ValueError("SCHEDULER_PAYLOAD_MODE must be measured or simulated_fixed")
MODEL_DIR = Path(os.environ.get("SCHEDULER_SWIN_MODEL_DIR", PROJECT_ROOT / "models/SwinJSCC"))
UPSTREAM_ROOT = PROJECT_ROOT / "external/SwinJSCC"
SCHEDULER_SETTINGS = load_scheduler_config(PROJECT_ROOT / "INFRA/config/semantic_scheduler_demo.yaml")
UE_PROFILES = load_ue_profiles(PROJECT_ROOT / "INFRA/config/ue_profiles.yaml")
DQN_SETTINGS = yaml.safe_load((PROJECT_ROOT / "Scheduler/config/semantic_dqn.yaml").read_text(encoding="utf-8"))
if DQN_SETTINGS["episodes"] != SCHEDULER_SETTINGS.train_episodes_per_seed:
    raise ValueError("DQN and stream episode counts disagree")
(RUN_ROOT / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2), encoding="utf-8")
print({"run_id": RUN_ID, "output": str(RUN_ROOT), "payload_mode": PAYLOAD_MODE})

## 3. Experiment-ready prototype dataset gate

Attach the private prototype dataset and set `SCHEDULER_MANIFEST` if
needed. It contains 60 unique images, six selection strata, 30
CLIP-proposed evidence pairs, and a 36/12/12 group-safe split.
P7=1 and the evidence pairs are prototype assumptions, not reviewed truth.

In [ ]:
MANIFEST_PATH = resolve_ladi_manifest(MANIFEST_PATH, RUN_ROOT / "input_bundle")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Prototype manifest missing: {MANIFEST_PATH}. Attach the private LADI dataset "+
        "or set SCHEDULER_MANIFEST before running this notebook.")
reviewed_rows = validate_ladi60_manifest(MANIFEST_PATH, require_reviewed=True)
frame = pd.DataFrame(reviewed_rows)
display(frame.groupby(["split", "selection_stratum"]).size().unstack(fill_value=0))
sheet = render_contact_sheet(MANIFEST_PATH, RUN_ROOT / "ladi60_contact_sheet.png")
from PIL import Image
display(Image.open(sheet))
print("All 60 hashes and fixed prototype annotations validated; no human-review claim is made.")

## 4. Extract CLIP features and cached image-classifier predictions once

In [ ]:
cache = prepare_feature_cache(MANIFEST_PATH, RUN_ROOT / "cache",
    include_ladi_predictions=True, device=DEVICE)
feature_rows = [json.loads(line) for line in cache["features"].read_text(encoding="utf-8").splitlines()]
extraction = pd.DataFrame({"image_id": row["image_id"],
                           "latency_ms": row["extraction_latency_ms"]} for row in feature_rows)
display(extraction.describe(include="all"))
print("Cached image embeddings are internal to INFRA; policies see five values and masks only.")

## 5. Cache payload cost once

Measured mode runs the existing checkpoint and records complex channel
uses separately from latent storage bytes. The checkpoint demands its
exported Python/PyTorch runtime; an incompatibility stops this cell.
`simulated_fixed` is only for an explicitly labelled infrastructure smoke run.

In [ ]:
if PAYLOAD_MODE == "simulated_fixed":
    PAYLOAD_COSTS = simulated_fixed_payload_costs(
        MANIFEST_PATH, RUN_ROOT / "payload" / "payload_costs.csv")
else:
    supplied_costs = os.environ.get("SCHEDULER_PAYLOAD_COSTS_CSV")
    if not supplied_costs:
        attached_costs = MANIFEST_PATH.parent / "payload_costs.csv"
        if not attached_costs.is_file():
            attached_costs = Path("/kaggle/input/semantic-scheduler-measured-costs/payload_costs.csv")
        supplied_costs = str(attached_costs) if attached_costs.is_file() else None
    if supplied_costs:
        PAYLOAD_COSTS = Path(supplied_costs)
        if not PAYLOAD_COSTS.is_file():
            raise FileNotFoundError(PAYLOAD_COSTS)
    else:
        PAYLOAD_COSTS = cache_swinjscc_payloads(
            reviewed_rows, RUN_ROOT / "payload", model_dir=MODEL_DIR,
            upstream_root=UPSTREAM_ROOT, device=DEVICE)
cost_frame = pd.read_csv(PAYLOAD_COSTS)
if cost_frame.resource_unit.nunique() != 1:
    raise ValueError("A run cannot mix resource units")
if PAYLOAD_MODE == "measured" and cost_frame.resource_unit.iloc[0] != "complex_channel_uses":
    raise ValueError("Measured mode requires checkpoint-derived channel uses")
display(cost_frame.head())

## 6. Freeze streams, UE profiles, deadlines, and transport tapes

In [ ]:
catalogue = load_catalog(MANIFEST_PATH, cache["features"], PAYLOAD_COSTS)
suites = build_suites_from_manifest(catalogue, MANIFEST_PATH,
                                     config=SCHEDULER_SETTINGS,
                                     ue_profiles=UE_PROFILES)
unit = catalogue[0].candidate.resource_unit
service_rate = max(item.candidate.resource_units for item in catalogue) / 10.0
budget = ResourceBudgetSemantic(service_rate, resource_unit=unit)
print({"train_episodes_per_seed": 300, "validation_episodes": len(suites["validation"]),
       "jobs_per_train_episode": len(suites["train_by_seed"][42][0].jobs),
       "service_rate_units_per_ms": service_rate, "resource_unit": unit})
episode_file = RUN_ROOT / "train_episodes.jsonl"
with episode_file.open("w", encoding="utf-8") as handle:
    for seed, episodes in suites["train_by_seed"].items():
        for episode in episodes:
            compact = {"episode_id": episode.episode_id, "seed": episode.seed,
                       "jobs": [{"job_id": job.candidate.job_id,
                                 "image_id": job.candidate.image_id,
                                 "ue_id": job.candidate.ue_id,
                                 "arrival_ms": job.candidate.arrival_ms,
                                 "deadline_ms": job.candidate.deadline_ms,
                                 "transport_success": job.transport_success}
                                for job in episode.jobs]}
            handle.write(json.dumps(compact) + "\n")

## 7. Fit the two supervised reference policies on 36 training images

In [ ]:
split_by_image = {row["image_id"]: row["split"] for row in reviewed_rows}
train_jobs = [job for job in catalogue if split_by_image[job.candidate.image_id] == "train"]
resolved_config = {
    "scheduler": asdict(SCHEDULER_SETTINGS), "ue_profiles": [asdict(p) for p in UE_PROFILES],
    "dqn": DQN_SETTINGS, "payload_mode": PAYLOAD_MODE,
    "service_rate_units_per_ms": service_rate, "resource_unit": unit,
    "seeds": [42, 43, 44],
}
(RUN_ROOT / "resolved_config.yaml").write_text(
    yaml.safe_dump(resolved_config, sort_keys=True), encoding="utf-8")
measured_checkpoint_sha = (str(cost_frame.checkpoint_sha256.iloc[0])
    if PAYLOAD_MODE == "measured" else None)
provenance = run_provenance(PROJECT_ROOT, MANIFEST_PATH,
    config=resolved_config, checkpoint_sha256=measured_checkpoint_sha)
if provenance["git_commit"] != SOURCE_COMMIT or provenance["git_dirty"]:
    raise RuntimeError("Training checkout changed after the clean Git sync")
models_dir = RUN_ROOT / "models"
supervised = fit_supervised_models(train_jobs, models_dir, provenance=provenance)
print({name: str(path) for name, path in supervised.items()})

## 8. Train three small standard DQNs (two-hour training cap)

Each action selects one eligible whole latent. Replay uses a target
network, masked Bellman targets, Huber loss and 459→64→32→25 layers.
Validation uses fixed streams and normalized semantic-utility AUC.
Each seed has a 40-minute wall-clock cap. All three seeds together
therefore have a two-hour training cap; extraction and evaluation
run outside that training budget.

In [ ]:
dqn_config = DQNConfig(**DQN_SETTINGS)
dqn_manifests = train_dqn_seeds(
    suites["train_by_seed"], suites["validation"], budget, models_dir,
    config=dqn_config, provenance=provenance, device=DEVICE)
training_rows = []
for seed in (42, 43, 44):
    path = models_dir / f"dqn_seed{seed}_history.json"
    training_rows.extend(json.loads(path.read_text(encoding="utf-8")))
training_frame = pd.DataFrame(training_rows)
training_frame.to_csv(RUN_ROOT / "training_metrics.csv", index=False)
display(training_frame.groupby("seed").tail(1))

## 9. Diagnostics and durable artifact manifest

In [ ]:
plots = RUN_ROOT / "plots"
plots.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for seed, group in training_frame.groupby("seed"):
    axes[0].plot(group.episode, group.reward, label=str(seed), alpha=.8)
    axes[1].plot(group.episode, group.loss, label=str(seed), alpha=.8)
    observed = group.dropna(subset=["validation_utility_auc"])
    axes[2].plot(observed.episode, observed.validation_utility_auc, marker="o", label=str(seed))
for axis, title in zip(axes, ("Episode reward", "Huber loss", "Validation utility AUC")):
    axis.set_title(title); axis.set_xlabel("Episode"); axis.grid(alpha=.2)
axes[0].legend(title="Seed")
fig.tight_layout(); fig.savefig(plots / "training_diagnostics.png", dpi=170)
display(fig); plt.close(fig)
action_counts = np.sum([json.loads((models_dir / f"dqn_seed{seed}_actions.json")
                        .read_text(encoding="utf-8"))["action_index_counts"]
                        for seed in (42, 43, 44)], axis=0)
fig, axis = plt.subplots(figsize=(10, 3))
axis.bar(np.arange(25), action_counts, color="#13678A")
axis.set(xlabel="Eligible candidate index at decision", ylabel="Selections",
         title="DQN action-index distribution across three seeds")
fig.tight_layout(); fig.savefig(plots / "action_distribution.png", dpi=170)
display(fig); plt.close(fig)

import shutil
shutil.copy2(MANIFEST_PATH, RUN_ROOT / "dataset_manifest.csv")
shutil.copy2(PAYLOAD_COSTS, RUN_ROOT / "payload_costs.csv")
shutil.copy2(cache["features"], RUN_ROOT / "feature_cache.jsonl")
shutil.copy2(cache["predictions"], RUN_ROOT / "ladi_predictions.json")
def file_hash(path):
    return sha256(Path(path).read_bytes()).hexdigest()
required = ["resolved_config.yaml", "dataset_manifest.csv", "payload_costs.csv", "feature_cache.jsonl",
            "ladi_predictions.json", "models/ordinal_classifier.npz",
            "models/pairwise_ranker.npz"] + [f"models/dqn_seed{seed}_best.npz" for seed in (42, 43, 44)]
artifact_hashes = {name: file_hash(RUN_ROOT / name) for name in required}
run_manifest = {
    **provenance, "run_id": RUN_ID, "status": "trained_on_prototype_fixed_manifest",
    "payload_mode": PAYLOAD_MODE, "resource_unit": unit,
    "service_rate_units_per_ms": service_rate,
    "artifacts_sha256": artifact_hashes,
    "dqn_best_validation": {str(seed): data["best_validation_utility_auc"]
                             for seed, data in dqn_manifests.items()},
}
(RUN_ROOT / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True), encoding="utf-8")
print("Persistent run:", RUN_ROOT)
print("Kaggle CLI: kaggle kernels output <username>/semantic-scheduler-train -p downloaded/train-run")

## 10. Evaluate every policy in this training run

The remaining cells run the same comparison and faculty walkthrough
as notebook 2 against the artifacts just produced. They save results
under this run directory; no second Kaggle job is needed.

In [ ]:
from dataclasses import asdict, replace
from INFRA.experiments.semantic_scheduler_experiment import (
    load_catalog, build_suites_from_manifest, evaluate_policy_suite)
from INFRA.semantic_streams import with_transport_probability
from Scheduler.semantic_policies import NumpyDQNPolicy
from INFRA.event_simulation import SemanticSchedulingEnvironment
from SemParam.semantic_header import SemanticSchedulingHeaderV2Codec

TRAIN_ROOT = RUN_ROOT
EVAL_ID = RUN_ID + "-evaluation"
EVAL_ROOT = RUN_ROOT / "evaluation"
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
training = run_manifest
print("Evaluation in the same run:", EVAL_ROOT)

## 3. Rebuild fixed streams and load every policy from one registry

In [ ]:
catalogue = load_catalog(MANIFEST_PATH, TRAIN_ROOT / "feature_cache.jsonl",
                         TRAIN_ROOT / "payload_costs.csv")
scheduler_settings = load_scheduler_config(PROJECT_ROOT / "INFRA/config/semantic_scheduler_demo.yaml")
ue_profiles = load_ue_profiles(PROJECT_ROOT / "INFRA/config/ue_profiles.yaml")
if resolved_config["scheduler"] != asdict(scheduler_settings):
    raise RuntimeError("Latest branch scheduler config differs from the training run")
if resolved_config["ue_profiles"] != [asdict(p) for p in ue_profiles]:
    raise RuntimeError("Latest branch UE profiles differ from the training run")
suites = build_suites_from_manifest(catalogue, MANIFEST_PATH,
    config=scheduler_settings, ue_profiles=ue_profiles)
budget = ResourceBudgetSemantic(training["service_rate_units_per_ms"],
                                resource_unit=training["resource_unit"])
priority_by_image = {job.candidate.image_id: job.base_priority for job in catalogue}
predictions = json.loads((TRAIN_ROOT / "ladi_predictions.json").read_text(encoding="utf-8"))
def model_path(filename):
    nested = TRAIN_ROOT / "models" / filename
    return nested if nested.is_file() else TRAIN_ROOT / filename
artifacts = {
    "feature_ordinal_classifier": model_path("ordinal_classifier.npz"),
    "pairwise_linear_ranker": model_path("pairwise_ranker.npz"),
    "dqn": model_path("dqn_seed42_best.npz"),
}
policy_names = (
    "fifo", "ue_round_robin", "fixed_priority", "edf", "priority_edf",
    "sjf", "aging", "semantic_greedy", "ladi_classifier_priority",
    "feature_ordinal_classifier", "pairwise_linear_ranker", "dqn",
    "privileged_marginal_utility")
allowed_inputs = pd.DataFrame([
    {"policy": name, "observes_privileged_rubric_priority": name in
     ("fixed_priority", "priority_edf", "privileged_marginal_utility"),
     "observes_evaluator_evidence_group": name == "privileged_marginal_utility",
     "observes_image_labels": False, "observes_semantic_features": name in
     ("semantic_greedy", "feature_ordinal_classifier", "pairwise_linear_ranker", "dqn")}
    for name in policy_names])
display(allowed_inputs)
print("The privileged one-step utility policy is a reference, not an upper bound on the optimal schedule.")

## 4. Fair test tracks

Each policy receives the same episode and pre-generated transport
result for every job. Image-disjoint test streams and closed-catalogue
sequence streams are reported separately. The latter does not test
unseen-image generalization.

In [ ]:
all_metrics = []
for track, episodes in (("image_disjoint", suites["image_disjoint"]),
                        ("closed_catalogue", suites["closed_catalogue"])):
    _, rows = evaluate_policy_suite(episodes, budget,
        priority_by_image=priority_by_image, predictions_by_image=predictions,
        model_artifacts=artifacts, policy_names=policy_names)
    all_metrics.extend({"track": track, "transport_p": 1.0, **row} for row in rows)
robustness = tuple(with_transport_probability(episode, .9)
                   for episode in suites["image_disjoint"])
_, robust_rows = evaluate_policy_suite(robustness, budget,
    priority_by_image=priority_by_image, predictions_by_image=predictions,
    model_artifacts=artifacts, policy_names=policy_names)
all_metrics.extend({"track": "image_disjoint_robustness", "transport_p": .9, **row}
                   for row in robust_rows)
metrics = pd.DataFrame(all_metrics)
metrics.to_csv(EVAL_ROOT / "all_policy_metrics.csv", index=False)
display(metrics.groupby(["track", "policy"])[["normalized_utility_auc",
    "critical_on_time_rate", "deadline_miss_rate", "weighted_priority_inversion",
    "jain_fairness"]].mean().round(3))

## 5. Uncertainty, cost sensitivity, and input-masking sensitivity

The semantic-greedy P2/P3 row is an automatic-feature comparator.
Masking inputs to classifier, ranker, or DQN models trained with all
five features is an out-of-distribution stress test. It is **not**
evidence for how separately retrained automatic-only models perform.

In [ ]:
rng = np.random.default_rng(2026)
def bootstrap_interval(values, repetitions=1000):
    clean = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(clean) == 0:
        return (np.nan, np.nan, np.nan)
    means = np.mean(rng.choice(clean, size=(repetitions, len(clean)), replace=True), axis=1)
    return (float(clean.mean()), float(np.quantile(means, .025)), float(np.quantile(means, .975)))
ci_rows = []
for (track, policy), group in metrics.groupby(["track", "policy"]):
    center, lower, upper = bootstrap_interval(group.normalized_utility_auc)
    ci_rows.append({"track": track, "policy": policy, "mean_auc": center,
                    "std_auc": float(group.normalized_utility_auc.std()),
                    "ci95_low": lower, "ci95_high": upper})
ci = pd.DataFrame(ci_rows)
ci.to_csv(EVAL_ROOT / "utility_auc_bootstrap_ci.csv", index=False)
display(ci[ci.track == "image_disjoint"].sort_values("mean_auc", ascending=False))

from INFRA.semantic_contracts import FEATURE_NAMES
ablations = {
    "full_oracle_assisted": FEATURE_NAMES,
    "p2_p3_automatic_only": ("context_relevance", "delivered_history_novelty"),
    "without_novelty": tuple(name for name in FEATURE_NAMES if name != "delivered_history_novelty"),
    "network_only": (),
}
ablation_rows = []
for label, fields in ablations.items():
    _, rows = evaluate_policy_suite(suites["image_disjoint"][:30], budget,
        priority_by_image=priority_by_image, predictions_by_image=predictions,
        model_artifacts=artifacts,
        policy_names=("semantic_greedy", "feature_ordinal_classifier",
                      "pairwise_linear_ranker", "dqn"),
        visible_features=tuple(fields))
    ablation_rows.extend({"ablation": label, **row} for row in rows)
ablation = pd.DataFrame(ablation_rows)
ablation.to_csv(EVAL_ROOT / "feature_ablation_metrics.csv", index=False)
display(ablation.groupby(["ablation", "policy"]).normalized_utility_auc.mean().unstack().round(3))
print("Input masking leaves reward/deadlines fixed. Learned-policy rows are OOD stress tests, not retrained ablations.")
sensitivity_rows = []
for coefficient in (0.0, 0.1, 0.2):
    _, rows = evaluate_policy_suite(suites["image_disjoint"][:30], budget,
        priority_by_image=priority_by_image, predictions_by_image=predictions,
        model_artifacts=artifacts,
        policy_names=("fifo", "semantic_greedy", "dqn"),
        cost_coefficient=coefficient)
    sensitivity_rows.extend({"cost_coefficient": coefficient, **row} for row in rows)
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(EVAL_ROOT / "fixed_policy_cost_sensitivity.csv", index=False)
display(sensitivity.groupby(["cost_coefficient", "policy"])
        [["episode_reward", "normalized_utility_auc"]].mean().round(3))
print("This is fixed-policy reward sensitivity. Only the 0.1 coefficient trained the DQN; retrain to test policy adaptation.")

## 5b. Ranking agreement on an all-at-once static snapshot

In [ ]:
from INFRA.semantic_evaluator import static_ranking_metrics
static_demo = replace(suites["faculty_demo"][0], episode_id="static-faculty-demo",
    jobs=tuple(replace(job, candidate=replace(job.candidate,
        arrival_ms=0.0, deadline_ms=1900.0))
        for job in suites["faculty_demo"][0].jobs))
static_results, _ = evaluate_policy_suite((static_demo,), budget,
    priority_by_image=priority_by_image, predictions_by_image=predictions,
    model_artifacts=artifacts, policy_names=policy_names, retain_results=True)
rank_frame = pd.DataFrame({"policy": result["policy"],
    **static_ranking_metrics(result)} for result in static_results)
rank_frame.to_csv(EVAL_ROOT / "static_ranking_metrics.csv", index=False)
display(rank_frame.sort_values("ndcg", ascending=False))
print("These ranking metrics apply only to the static 25-job fixture, not staggered UE arrivals.")

## 6. Plots for the faculty comparison

In [ ]:
plot_dir = EVAL_ROOT / "plots"; plot_dir.mkdir(exist_ok=True)
subset = metrics[metrics.track == "image_disjoint"]
means = subset.groupby("policy").mean(numeric_only=True)
order = means.normalized_utility_auc.sort_values(ascending=False).index
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
means.loc[order, "normalized_utility_auc"].plot.bar(ax=axes[0,0], color="#13678A")
axes[0,0].set_title("Semantic utility over time (AUC)")
means.loc[order, "critical_on_time_rate"].plot.bar(ax=axes[0,1], color="#11A09A")
axes[0,1].set_title("Critical/emergency delivered on time")
means.loc[order, "weighted_priority_inversion"].plot.bar(ax=axes[1,0], color="#D98432")
axes[1,0].set_title("Eligible priority inversion severity")
axes[1,1].scatter(means.mean_policy_inference_ms, means.normalized_utility_auc)
for name in means.index:
    axes[1,1].annotate(name, (means.loc[name,"mean_policy_inference_ms"],
                               means.loc[name,"normalized_utility_auc"]), fontsize=8)
axes[1,1].set(xlabel="Policy inference (ms)", ylabel="Utility AUC",
               title="Inference cost versus utility")
for axis in axes.flat: axis.grid(alpha=.15)
fig.tight_layout(); fig.savefig(plot_dir / "policy_comparison.png", dpi=170)
display(fig); plt.close(fig)

## 7. Faculty scenario: 25 images, five UEs, one visible decision

The table exposes the complete queue snapshot at a decision point:
five values, availability masks, payload units, deadlines, score and
selected action. The 24-byte header is encoded and CRC-checked here.

In [ ]:
demo = suites["faculty_demo"][0]
_, demo_metrics = evaluate_policy_suite((demo,), budget,
    priority_by_image=priority_by_image, predictions_by_image=predictions,
    model_artifacts=artifacts, policy_names=policy_names)
demo_results, _ = evaluate_policy_suite((demo,), budget,
    priority_by_image=priority_by_image, predictions_by_image=predictions,
    model_artifacts=artifacts, policy_names=policy_names, retain_results=True)
display(pd.DataFrame(demo_metrics).sort_values("normalized_utility_auc", ascending=False))
image_paths = {row["image_id"]: row["image_path"] for row in reviewed_rows}
from PIL import Image
fig, axes = plt.subplots(5, 5, figsize=(15, 15))
for axis, job in zip(axes.flat, demo.jobs):
    with Image.open(image_paths[job.candidate.image_id]) as opened:
        axis.imshow(opened.convert("RGB"))
    axis.set_title(f"{job.candidate.ue_id} | {job.candidate.arrival_ms:.0f} ms", fontsize=8)
    axis.axis("off")
fig.suptitle("25 distinct LADI images assigned to five simulated UEs")
fig.tight_layout(); fig.savefig(plot_dir / "faculty_25_inputs.png", dpi=130)
display(fig); plt.close(fig)
chosen_demo = next(result for result in demo_results if result["policy"] == "dqn")
first = next(row for row in chosen_demo["decision_trace"] if len(row["candidate_snapshot"]) > 1)
candidates = pd.DataFrame(first["candidate_snapshot"])
feature_names = ("P2 relevance", "P3 novelty", "P5 impact", "P6 urgency", "P7 sufficiency")
for index, name in enumerate(feature_names):
    candidates[name] = candidates.features.map(lambda values: values[index])
    candidates[name + " available"] = candidates.availability.map(lambda mask: mask[index])
display(candidates.drop(columns=["features", "raw_features", "availability"]))
print("DQN selected:", first["job_id"], "at", first["time_ms"], "ms; reward:", first["reward"])
wire = bytes.fromhex(first["header_hex"])
decoded_header = SemanticSchedulingHeaderV2Codec().decode(wire)
assert len(wire) == first["header_size_bytes"] == 24
print("Logical control header:", len(wire), "bytes /", len(wire)*8,
      "bits; hex:", wire.hex())
print("Decoded decision-time fields:", decoded_header)

dqn = NumpyDQNPolicy(artifacts["dqn"])
demo_environment = SemanticSchedulingEnvironment(demo, budget)
for recorded_decision in chosen_demo["decision_trace"]:
    snapshot = demo_environment.snapshot()
    if recorded_decision is first:
        break
    demo_environment.step(recorded_decision["action_index"])
assert snapshot.now_ms == first["time_ms"]
q_values, valid_mask = dqn.q_values(snapshot, budget)
assert int(np.argmax(np.where(valid_mask, q_values, -np.inf))) == first["action_index"]
display(pd.DataFrame({"job_id": [job.job_id for job in snapshot.jobs],
    "eligible": valid_mask[:len(snapshot.jobs)],
    "Q_value": q_values[:len(snapshot.jobs)]}).sort_values("Q_value", ascending=False))

## 8. Timeline, novelty change, and selected real checkpoint replay

In [ ]:
demo_trace = []
for result in demo_results:
    for row in result["outcomes"]:
        demo_trace.append({"policy": result["policy"], **row})
timeline = pd.DataFrame(demo_trace)
timeline.to_csv(EVAL_ROOT / "faculty_timeline.csv", index=False)
display(timeline[timeline.policy == "dqn"].sort_values("started_ms")[
    ["image_id", "ue_id", "arrival_ms", "started_ms", "completed_ms",
     "status", "utility_delta"]])
novelty_rows = []
for decision in chosen_demo["decision_trace"]:
    for candidate in decision["candidate_snapshot"]:
        novelty_rows.append({"time_ms": decision["time_ms"],
            "history_version": decision["history_version"],
            "image_id": candidate["image_id"],
            "novelty": candidate["raw_features"][1],
            "novelty_available": candidate["availability"][1]})
novelty_frame = pd.DataFrame(novelty_rows)
changing = novelty_frame.groupby("image_id").history_version.nunique()
if (changing > 1).any():
    demonstration_image = changing[changing > 1].index[0]
    display(novelty_frame[novelty_frame.image_id == demonstration_image]
            .drop_duplicates(["history_version", "novelty"]))
else:
    print("No waiting image spanned two history versions in this stream.")
fig, axis = plt.subplots(figsize=(11,5))
for result in demo_results:
    times, values = zip(*result["utility_curve"])
    axis.step(times, values, where="post", label=result["policy"], alpha=.8)
axis.set(xlabel="Simulated time (ms)", ylabel="Accumulated semantic utility",
         title="Identical faculty stream, different scheduling decisions")
axis.grid(alpha=.2); axis.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(plot_dir / "faculty_utility_timeline.png", dpi=170)
display(fig); plt.close(fig)

from EncDec.integration.scheduler_replay import replay_selected_jobs
replay_status = {"status": "unavailable", "reason": "payload mode is simulated"}
if training["payload_mode"] == "measured":
    model_dir = Path(os.environ.get("SCHEDULER_SWIN_MODEL_DIR", PROJECT_ROOT / "models/SwinJSCC"))
    selected_ids = [row["image_id"] for row in chosen_demo["outcomes"]
                    if row["status"] in ("completed_on_time", "completed_late")][:3]
    try:
        attached_cache = TRAIN_ROOT / "payload" / "payload_costs.csv"
        if not attached_cache.is_file():
            from EncDec.integration.scheduler_replay import cache_swinjscc_payloads
            selected_rows = [row for row in reviewed_rows if row["image_id"] in selected_ids]
            attached_cache = cache_swinjscc_payloads(selected_rows,
                EVAL_ROOT / "selected_payload_cache", model_dir=model_dir,
                upstream_root=PROJECT_ROOT / "external/SwinJSCC", device=DEVICE)
        replay_path = replay_selected_jobs(selected_ids, image_paths,
            attached_cache, EVAL_ROOT / "replay",
            model_dir=model_dir, upstream_root=PROJECT_ROOT / "external/SwinJSCC",
            device=DEVICE)
        replay_status = {"status": "measured_checkpoint_inference", "csv": str(replay_path)}
        display(pd.read_csv(replay_path))
    except (FileNotFoundError, RuntimeError, ValueError) as error:
        replay_status = {"status": "unavailable", "reason": str(error)}
(EVAL_ROOT / "replay_status.json").write_text(json.dumps(replay_status, indent=2), encoding="utf-8")
print("Codec replay:", replay_status)

## 9. Claim ledger and output persistence

In [ ]:
claims = pd.DataFrame([
    {"category":"Implemented", "component":"P2/P3 extraction, prototype-rubric P5/P6 and default P7, 24-byte header, policies, DQN, metrics"},
    {"category":"Simulated", "component":"UE arrivals, queues, deadlines, service duration, transport outcome"},
    {"category":"Measured when available", "component":"SwinJSCC checkpoint latency, channel uses, selected PSNR/SSIM"},
    {"category":"Proposed", "component":"Real 5G MAC/PHY control channel, RB allocation, distributed KB sync"},
])
display(claims)
claims.to_csv(EVAL_ROOT / "claim_ledger.csv", index=False)
summary = {"evaluation_id": EVAL_ID, "exact_commit": SOURCE_COMMIT,
           "training_run_id": training["run_id"], "payload_mode": training["payload_mode"],
           "image_disjoint_streams": len(suites["image_disjoint"]),
           "closed_catalogue_streams": len(suites["closed_catalogue"]),
           "replay": replay_status}
(EVAL_ROOT / "evaluation_manifest.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Persistent evaluation:", EVAL_ROOT)
print("Kaggle CLI: kaggle kernels output <username>/semantic-scheduler-evaluate -p downloaded/evaluation-run")